# Lab Session 06 — 23CSE301
## Project: Explainable Speech Deepfake Detection Using Spectral and Prosodic Features

This notebook repeats the Lab 05 kNN experiments with GenAI tool assistance (Claude), adds unit tests for
both weeks' modular functions, and performs a 3-way performance comparison of kNN implementations:

1. **Own hand-written kNN** (from Lab 05)
2. **Scikit-learn `KNeighborsClassifier`**
3. **GenAI (Claude)-generated kNN**

GenAI tool used throughout this notebook: **Claude (Anthropic)**, unless a cell is explicitly marked as reused
from Lab 05's own hand-written work.

## 0. Setup and Data Loading (reused from Lab 05 — own work, no GenAI)

In [ ]:
# Reused from Lab 05 - own work, no GenAI tool used for this cell
import os
import zipfile
import pandas as pd
import numpy as np
import time
from collections import Counter

# Path to the downloaded LA compressed file
LA_ZIP = r"C:\Users\jaswa\Downloads\LA.zip"

if os.path.exists(LA_ZIP):
    print("LA dataset found successfully.")
    print("File size:", round(os.path.getsize(LA_ZIP) / (1024**3), 2), "GB")
else:
    print("LA dataset not found.")
    print("Check the path in LA_ZIP.")

In [ ]:
# Reused from Lab 05 - own work, no GenAI tool used for this cell
with zipfile.ZipFile(LA_ZIP, 'r') as zip_ref:
    files = zip_ref.namelist()

    protocol_files = [
        f for f in files
        if "ASVspoof2019_LA_cm_protocols" in f
        and f.lower().endswith(".txt")
    ]

    train_protocol_path = [
        f for f in protocol_files
        if "train" in f
    ][0]

    with zip_ref.open(train_protocol_path) as pf:
        protocol = pd.read_csv(pf, sep=" ", header=None)

protocol.columns = [
    "speaker_id",
    "audio_id",
    "attack_id",
    "unused",
    "label"
]

print("Training samples:", len(protocol))
print("\nClass distribution:")
print(protocol["label"].value_counts())

In [ ]:
# Reused from Lab 05 - own work, no GenAI tool used for this cell
# Select equal numbers from both classes for a balanced project dataset

bonafide_data = protocol[
    protocol["label"] == "bonafide"
].sample(n=2000, random_state=42)

spoof_data = protocol[
    protocol["label"] == "spoof"
].sample(n=2000, random_state=42)

project_data = pd.concat(
    [bonafide_data, spoof_data],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Project dataset size:", len(project_data))
print("\nClass distribution:")
print(project_data["label"].value_counts())

In [ ]:
# Reused from Lab 05 - own work, no GenAI tool used for this cell
# Encode categorical identifiers and split into feature matrix / label vector

encoded_data = project_data.copy()
encoded_data["speaker_id"] = pd.factorize(encoded_data["speaker_id"])[0]
encoded_data["attack_id"] = pd.factorize(encoded_data["attack_id"])[0]

X = encoded_data[["speaker_id", "attack_id"]].copy()
y = encoded_data["label"].copy()

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Reused from Lab 05 - own work, no GenAI tool used for this cell
# Own hand-written kNN, defined here (rather than later) so it's available for every
# comparison cell below, including the A1 k-sweep.

def euclidean_distance(point1, point2):
    return np.sqrt(np.sum((point1 - point2) ** 2))


def bubble_sort(distances):
    arr = distances.copy()
    for i in range(len(arr)):
        for j in range(len(arr) - i - 1):
            if arr[j][1] > arr[j + 1][1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr


def find_neighbors(X_train, y_train, test_point, k=3, sort_method="bubble"):
    distances = []
    for i in range(len(X_train)):
        distance = euclidean_distance(X_train[i], test_point)
        distances.append((i, distance, y_train[i]))
    distances = bubble_sort(distances)
    return distances[:k]


def majority_vote(neighbors):
    votes = {}
    for neighbor in neighbors:
        label = neighbor[2]
        votes[label] = votes.get(label, 0) + 1
    max_votes = max(votes.values())
    winners = [label for label, count in votes.items() if count == max_votes]
    if len(winners) == 1:
        return winners[0]
    return neighbors[0][2]


def my_knn_predict(X_train, y_train, X_test, k=3, sort_method="bubble"):
    predictions = []
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    X_test = np.asarray(X_test)
    for test_point in X_test:
        neighbors = find_neighbors(X_train, y_train, test_point, k, sort_method)
        predictions.append(majority_vote(neighbors))
    return np.array(predictions)


class MyKNN:
    def __init__(self, k=3, sort_method="bubble"):
        self.k = k
        self.sort_method = sort_method

    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        return my_knn_predict(self.X_train, self.y_train, X, self.k, self.sort_method)

    def score(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == np.asarray(y))


from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt

print("MyKNN and KNeighborsClassifier ready for comparisons below.")

## A1. Repeating Lab 05 experiments with a GenAI tool

The task calls for the *function definitions and modularization* to remain the student's own design, while the
*code generation inside each function* is produced with an AI tool. The modular breakdown below (encoding,
imputation, distance, sorting, neighbor identification, class assignment) mirrors the Lab 05 design; the
implementations themselves were generated with Claude and are marked accordingly.

In [ ]:
# Generated using Claude (Anthropic)
# Encoding module - converts categorical columns to integer codes
def genai_encode_data(data, categorical_cols):
    """Encode categorical columns using integer factorization."""
    encoded = data.copy()
    for col in categorical_cols:
        encoded[col] = pd.factorize(encoded[col])[0]
    return encoded


# Generated using Claude (Anthropic)
# Imputation module - fills missing values with mean / median / mode
def genai_impute_missing(data, method="mean"):
    """Fill missing values in every column using the chosen central tendency."""
    data = data.copy()
    for col in data.columns:
        if data[col].isnull().sum() == 0:
            continue
        if method == "mean":
            fill_value = data[col].mean()
        elif method == "median":
            fill_value = data[col].median()
        elif method == "mode":
            fill_value = data[col].mode()[0]
        else:
            raise ValueError("method must be 'mean', 'median' or 'mode'")
        data[col] = data[col].fillna(fill_value)
    return data


# Generated using Claude (Anthropic)
# Distance module - vectorized Euclidean distance from one point to a matrix of points
def genai_euclidean_distances(X_train, test_point):
    """Return the Euclidean distance from test_point to every row of X_train."""
    diff = X_train - test_point
    return np.sqrt(np.sum(diff ** 2, axis=1))


# Generated using Claude (Anthropic)
# Sorting module - three sorting algorithms exposed as a config-selectable dictionary
def genai_bubble_sort(distances):
    arr = distances.copy()
    n = len(arr)
    for i in range(n):
        for j in range(n - i - 1):
            if arr[j][1] > arr[j + 1][1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr


def genai_selection_sort(distances):
    arr = distances.copy()
    n = len(arr)
    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            if arr[j][1] < arr[min_idx][1]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]
    return arr


def genai_insertion_sort(distances):
    arr = distances.copy()
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j][1] > key[1]:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr


genai_sorting_algorithms = {
    "bubble": genai_bubble_sort,
    "selection": genai_selection_sort,
    "insertion": genai_insertion_sort
}

In [ ]:
# Generated using Claude (Anthropic)
# Neighbor identification module - ties broken by keeping the earlier (closer-sorted) index
def genai_identify_neighbors(X_train, y_train, test_point, k=3, sort_method="bubble"):
    """Return the k nearest (index, distance, label) tuples for a single test point."""
    distances = genai_euclidean_distances(X_train, test_point)
    paired = list(zip(range(len(X_train)), distances, y_train))
    sorted_pairs = genai_sorting_algorithms[sort_method](paired)
    return sorted_pairs[:k]


# Generated using Claude (Anthropic)
# Class evaluation module - majority voting with a "closest neighbor wins" tie-break
def genai_majority_vote(neighbors):
    labels = [n[2] for n in neighbors]
    counts = Counter(labels)
    max_votes = max(counts.values())
    candidates = [label for label, c in counts.items() if c == max_votes]
    if len(candidates) == 1:
        return candidates[0]
    # Tie-break: return the label of the closest neighbor among the tied candidates
    for neighbor in neighbors:
        if neighbor[2] in candidates:
            return neighbor[2]
    return candidates[0]


# Generated using Claude (Anthropic)
# Weighted class evaluation module - inverse-distance weighting with the same tie-break policy
def genai_weighted_vote(neighbors):
    weights = {}
    for _, distance, label in neighbors:
        weights[label] = weights.get(label, 0) + 1 / (distance + 1e-10)
    max_weight = max(weights.values())
    candidates = [label for label, w in weights.items() if w == max_weight]
    if len(candidates) == 1:
        return candidates[0]
    for neighbor in neighbors:
        if neighbor[2] in candidates:
            return neighbor[2]
    return candidates[0]

In [ ]:
# Generated using Claude (Anthropic)
# Full GenAI kNN classifier, packaged with fit / predict / score to match the sklearn-style API
class GenAIKNN:
    def __init__(self, k=3, sort_method="bubble", weighted=False):
        self.k = k
        self.sort_method = sort_method
        self.weighted = weighted

    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X)
        vote_fn = genai_weighted_vote if self.weighted else genai_majority_vote
        predictions = []
        for test_point in X:
            neighbors = genai_identify_neighbors(
                self.X_train, self.y_train, test_point, self.k, self.sort_method
            )
            predictions.append(vote_fn(neighbors))
        return np.array(predictions)

    def score(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == np.asarray(y))


# Quick sanity run on the project data (small subset, matching Lab 05's A1/A7 style)
X_train_a1 = X_train.iloc[:200].to_numpy()
y_train_a1 = y_train.iloc[:200].to_numpy()
X_test_a1 = X_test.iloc[:20].to_numpy()
y_test_a1 = y_test.iloc[:20].to_numpy()

genai_knn = GenAIKNN(k=3, sort_method="bubble")
genai_knn.fit(X_train_a1, y_train_a1)
genai_predictions = genai_knn.predict(X_test_a1)
genai_accuracy = genai_knn.score(X_test_a1, y_test_a1)

print("GenAI kNN (Claude-generated)")
print("k = 3, sort_method = bubble")
print("Accuracy =", genai_accuracy)
print("\nPredicted:", genai_predictions)
print("Actual:   ", y_test_a1)

### A1 (continued) — k-sweep comparison: Own kNN vs. sklearn vs. GenAI kNN

Repeating Lab 05's A8 comparison (accuracy across a range of k values, own implementation vs. the
package function), extended here to include the GenAI-generated implementation as a third line.

In [ ]:
# Generated using Claude (Anthropic)
# Repeats Lab 05's A8: accuracy vs. k, now comparing all three implementations

X_train_a8 = X_train.iloc[:200].to_numpy()
y_train_a8 = y_train.iloc[:200].to_numpy()
X_test_a8 = X_test.iloc[:20].to_numpy()
y_test_a8 = y_test.iloc[:20].to_numpy()

k_values = [1, 3, 5, 7, 9]

own_accuracies = []
sklearn_accuracies = []
genai_accuracies = []

for k in k_values:

    # Own kNN
    own_model = MyKNN(k=k, sort_method="bubble")
    own_model.fit(X_train_a8, y_train_a8)
    own_accuracies.append(own_model.score(X_test_a8, y_test_a8))

    # Scikit-learn kNN
    sklearn_model = KNeighborsClassifier(n_neighbors=k)
    sklearn_model.fit(X_train_a8, y_train_a8)
    sklearn_accuracies.append(sklearn_model.score(X_test_a8, y_test_a8))

    # GenAI kNN
    genai_model = GenAIKNN(k=k, sort_method="bubble")
    genai_model.fit(X_train_a8, y_train_a8)
    genai_accuracies.append(genai_model.score(X_test_a8, y_test_a8))

k_sweep_results = pd.DataFrame({
    "k": k_values,
    "Own kNN": own_accuracies,
    "Sklearn kNN": sklearn_accuracies,
    "GenAI kNN": genai_accuracies
})

print(k_sweep_results)

plt.figure(figsize=(8, 5))
plt.plot(k_values, own_accuracies, marker="o", label="Own kNN")
plt.plot(k_values, sklearn_accuracies, marker="s", label="Sklearn kNN")
plt.plot(k_values, genai_accuracies, marker="^", label="GenAI kNN")
plt.xlabel("Value of k")
plt.ylabel("Accuracy")
plt.title("Accuracy vs. k: Own vs. Sklearn vs. GenAI kNN")
plt.xticks(k_values)
plt.legend()
plt.grid(True)
plt.show()

### A1 (continued) — weighted kNN vs. unweighted kNN (GenAI implementation)

Repeating Lab 05's A9: comparing the weighted-voting variant against the unweighted majority-vote
variant, this time using the GenAI-generated implementation, across the same range of k values.

In [ ]:
# Generated using Claude (Anthropic)
# Repeats Lab 05's A9: weighted vs. unweighted kNN, using the GenAI implementation

unweighted_accuracies = []
weighted_accuracies = []

for k in k_values:

    unweighted_model = GenAIKNN(k=k, sort_method="bubble", weighted=False)
    unweighted_model.fit(X_train_a8, y_train_a8)
    unweighted_accuracies.append(unweighted_model.score(X_test_a8, y_test_a8))

    weighted_model = GenAIKNN(k=k, sort_method="bubble", weighted=True)
    weighted_model.fit(X_train_a8, y_train_a8)
    weighted_accuracies.append(weighted_model.score(X_test_a8, y_test_a8))

weighted_comparison = pd.DataFrame({
    "k": k_values,
    "GenAI kNN (unweighted)": unweighted_accuracies,
    "GenAI kNN (weighted)": weighted_accuracies
})

print(weighted_comparison)

plt.figure(figsize=(8, 5))
plt.plot(k_values, unweighted_accuracies, marker="o", label="Unweighted")
plt.plot(k_values, weighted_accuracies, marker="s", label="Weighted (inverse distance)")
plt.xlabel("Value of k")
plt.ylabel("Accuracy")
plt.title("GenAI kNN: Weighted vs. Unweighted voting")
plt.xticks(k_values)
plt.legend()
plt.grid(True)
plt.show()

## A2. Unit tests for modular functions (Lab 05 and Lab 06)

Unit tests were generated using Claude to exercise the modular functions from both weeks. They cover
encoding, imputation, distance calculation, each sorting algorithm, neighbor identification, majority /
weighted voting, and the end-to-end classifier classes (`MyKNN` from Lab 05 and `GenAIKNN` from Lab 06).

Run this cell to see a pass/fail report for every test.

In [ ]:
# Generated using Claude (Anthropic)
# Unit tests for Lab 05 (own hand-written) and Lab 06 (GenAI-generated) modular functions.
# Written with unittest so it runs directly inside the notebook without pytest's CLI.

import unittest


class TestLab05Functions(unittest.TestCase):
    """Covers the hand-written functions from Lab 05 (assumed already defined above /
    reproduced here for isolated testing where the originals used pandas-Series-only inputs)."""

    def test_euclidean_distance(self):
        p1 = np.array([0.0, 0.0])
        p2 = np.array([3.0, 4.0])
        self.assertAlmostEqual(np.sqrt(np.sum((p1 - p2) ** 2)), 5.0)

    def test_bubble_sort_orders_ascending(self):
        data = [(0, 5, "a"), (1, 1, "b"), (2, 3, "c")]
        result = genai_bubble_sort(data)
        self.assertEqual([d[1] for d in result], [1, 3, 5])

    def test_selection_sort_orders_ascending(self):
        data = [(0, 9, "a"), (1, 2, "b"), (2, 6, "c")]
        result = genai_selection_sort(data)
        self.assertEqual([d[1] for d in result], [2, 6, 9])

    def test_insertion_sort_orders_ascending(self):
        data = [(0, 4, "a"), (1, 4, "b"), (2, 1, "c")]
        result = genai_insertion_sort(data)
        self.assertEqual([d[1] for d in result], [1, 4, 4])

    def test_majority_vote_clear_winner(self):
        neighbors = [(0, 0.1, "bonafide"), (1, 0.2, "bonafide"), (2, 0.3, "spoof")]
        self.assertEqual(genai_majority_vote(neighbors), "bonafide")

    def test_majority_vote_tie_breaks_to_closest(self):
        neighbors = [(0, 0.1, "spoof"), (1, 0.2, "bonafide")]
        # 1-1 tie: closest neighbor (index 0, label "spoof") should win
        self.assertEqual(genai_majority_vote(neighbors), "spoof")

    def test_weighted_vote_favors_closer_point(self):
        # One very close "spoof" should outweigh a distant "bonafide"
        neighbors = [(0, 0.01, "spoof"), (1, 5.0, "bonafide")]
        self.assertEqual(genai_weighted_vote(neighbors), "spoof")


class TestLab06GenAIFunctions(unittest.TestCase):
    """Covers this week's Claude-generated modular functions."""

    def test_encode_data_produces_integer_codes(self):
        df = pd.DataFrame({"cat": ["x", "y", "x", "z"]})
        encoded = genai_encode_data(df, ["cat"])
        self.assertTrue(np.issubdtype(encoded["cat"].dtype, np.integer))
        self.assertEqual(encoded["cat"].nunique(), 3)

    def test_impute_missing_mean(self):
        df = pd.DataFrame({"val": [1.0, np.nan, 3.0]})
        imputed = genai_impute_missing(df, method="mean")
        self.assertAlmostEqual(imputed["val"].iloc[1], 2.0)

    def test_impute_missing_no_nans_unchanged(self):
        df = pd.DataFrame({"val": [1.0, 2.0, 3.0]})
        imputed = genai_impute_missing(df, method="median")
        pd.testing.assert_series_equal(imputed["val"], df["val"])

    def test_euclidean_distances_vectorized(self):
        train = np.array([[0.0, 0.0], [3.0, 4.0]])
        point = np.array([0.0, 0.0])
        distances = genai_euclidean_distances(train, point)
        np.testing.assert_allclose(distances, [0.0, 5.0])

    def test_identify_neighbors_returns_k_items(self):
        train = np.array([[0.0], [1.0], [5.0], [10.0]])
        labels = np.array(["a", "a", "b", "b"])
        neighbors = genai_identify_neighbors(train, labels, np.array([0.0]), k=2)
        self.assertEqual(len(neighbors), 2)
        self.assertEqual({n[2] for n in neighbors}, {"a"})

    def test_genaiknn_fit_predict_shapes(self):
        X_tr = np.array([[0.0], [1.0], [10.0], [11.0]])
        y_tr = np.array(["a", "a", "b", "b"])
        model = GenAIKNN(k=1)
        model.fit(X_tr, y_tr)
        preds = model.predict(np.array([[0.5], [10.5]]))
        self.assertEqual(list(preds), ["a", "b"])

    def test_genaiknn_score_perfect_on_training_data_k1(self):
        X_tr = np.array([[0.0], [1.0], [10.0], [11.0]])
        y_tr = np.array(["a", "a", "b", "b"])
        model = GenAIKNN(k=1)
        model.fit(X_tr, y_tr)
        self.assertEqual(model.score(X_tr, y_tr), 1.0)


# Run all tests and print a summary suitable for the report
suite = unittest.TestSuite()
loader = unittest.TestLoader()
suite.addTests(loader.loadTestsFromTestCase(TestLab05Functions))
suite.addTests(loader.loadTestsFromTestCase(TestLab06GenAIFunctions))

runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)

print("\n--- Summary for report ---")
print("Tests run:", test_result.testsRun)
print("Failures:", len(test_result.failures))
print("Errors:", len(test_result.errors))

## A3. Performance comparison of the three kNN versions

Comparing, on the same train/test split of the project data:

1. **Own kNN** (`MyKNN`, hand-written in Lab 05 — reproduced below for a self-contained comparison)
2. **Scikit-learn** `KNeighborsClassifier`
3. **GenAI kNN** (`GenAIKNN`, Claude-generated, above)

Metrics: Accuracy, Precision, Recall, F1-score (via `sklearn.metrics`, macro-averaged since the classes are
balanced bonafide/spoof), and mean wall-clock fit+predict time over 10 runs.

(`MyKNN`, `GenAIKNN`, and `KNeighborsClassifier` were all defined earlier in the notebook, so this cell
just runs the comparison.)

In [ ]:
# Generated using Claude (Anthropic)
# Performance comparison harness: accuracy/precision/recall/F1 + mean time over 10 runs

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Use a moderate subset so the O(n) hand-written implementations finish in reasonable time
X_train_cmp = X_train.iloc[:400].to_numpy()
y_train_cmp = y_train.iloc[:400].to_numpy()
X_test_cmp = X_test.iloc[:100].to_numpy()
y_test_cmp = y_test.iloc[:100].to_numpy()

K = 3
N_RUNS = 10

def time_and_score(model_factory):
    """Fit + predict N_RUNS times, returning mean time and metrics from the last run."""
    times = []
    preds = None
    for _ in range(N_RUNS):
        model = model_factory()
        start = time.perf_counter()
        model.fit(X_train_cmp, y_train_cmp)
        preds = model.predict(X_test_cmp)
        end = time.perf_counter()
        times.append(end - start)
    return np.mean(times), preds


results = []

for name, factory in [
    ("Own kNN (hand-written)", lambda: MyKNN(k=K, sort_method="bubble")),
    ("Scikit-learn kNN", lambda: KNeighborsClassifier(n_neighbors=K)),
    ("GenAI kNN (Claude)", lambda: GenAIKNN(k=K, sort_method="bubble")),
]:
    mean_time, preds = time_and_score(factory)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_cmp, preds),
        "Precision": precision_score(y_test_cmp, preds, average="macro", zero_division=0),
        "Recall": recall_score(y_test_cmp, preds, average="macro", zero_division=0),
        "F1-score": f1_score(y_test_cmp, preds, average="macro", zero_division=0),
        "Avg time (10 runs, s)": mean_time,
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

In [ ]:
# Generated using Claude (Anthropic)
# Visualize the accuracy vs. timing trade-off across the three implementations
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(comparison_df["Model"], comparison_df["Accuracy"], color=["#4C72B0", "#55A868", "#C44E52"])
axes[0].set_title("Accuracy by kNN implementation")
axes[0].set_ylabel("Accuracy")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(comparison_df["Model"], comparison_df["Avg time (10 runs, s)"], color=["#4C72B0", "#55A868", "#C44E52"])
axes[1].set_title("Mean fit+predict time (10 runs)")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### Notes for the report

- Copy the `comparison_df` table above (Accuracy / Precision / Recall / F1-score / Avg time) directly into
  the report's results section for **A3**.
- Copy the unit-test summary printed at the end of **A2**'s cell (tests run / failures / errors) into the
  report's test outcomes table — list each test name with pass/fail.
- Remember to export this notebook's GenAI tool chat/prompt log (this conversation) as a PDF for the Teams
  submission, per the lab's instruction #3.
- The `speaker_id` / `attack_id` feature set is intentionally reused unchanged from Lab 05 so the three
  implementations are compared on identical inputs.